In [55]:
import numpy as np
import pandas as pd
import neurokit2 as nk

In [56]:
fs = 8

Imported reshape function as it will be needed to make row-wise descriptives

In [57]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'EDA', window_size = 60, stride_size = 15, hertz=8, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [58]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")
eda = pd.read_csv("../Data/wesad_eda/filtered_eda_signals.csv")

In [59]:
print(tonic.shape)
print(phasic.shape)
print(eda.shape)

(20892, 30)
(20892, 30)
(20892, 30)


This section is just applying the reshape function and verifying the outputs

In [60]:
eda.head()

,S15_EDA,S15_Label,S7_EDA,S7_Label,S3_EDA,S3_Label,S2_EDA,S2_Label,S14_EDA,S14_Label,...,S4_EDA,S4_Label,S13_EDA,S13_Label,S5_EDA,S5_Label,S9_EDA,S9_Label,S6_EDA,S6_Label
0,0.000000,0.0,0.000000,0.0,0.016044,0.0,0.097471,0.0,0.011914,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.037751,0.0,0.033865,0
1,0.073305,0.0,0.485077,0.0,0.178527,0.0,0.492093,0.0,0.175268,0.0,...,0.031140,0.0,0.366758,0.0,0.065186,0.0,0.144301,0.0,0.199618,0
2,0.116746,0.0,0.854691,0.0,0.319746,0.0,0.769926,0.0,0.278055,0.0,...,0.053885,0.0,0.628746,0.0,0.109112,0.0,0.224244,0.0,0.323051,0
3,0.083819,0.0,0.729083,0.0,0.296441,0.0,0.620082,0.0,0.191334,0.0,...,0.044419,0.0,0.511660,0.0,0.085603,0.0,0.197046,0.0,0.275535,0
4,0.062718,0.0,0.609167,0.0,0.256728,0.0,0.491979,0.0,0.120382,0.0,...,0.035439,0.0,0.421939,0.0,0.063158,0.0,0.172040,0.0,0.231554,0


In [61]:
tonic.head()

,S15_Tonic,S15_Label,S7_Tonic,S7_Label,S3_Tonic,S3_Label,S2_Tonic,S2_Label,S14_Tonic,S14_Label,...,S4_Tonic,S4_Label,S13_Tonic,S13_Label,S5_Tonic,S5_Label,S9_Tonic,S9_Label,S6_Tonic,S6_Label
0,0.059960,0.0,0.298492,0.0,0.117891,0.0,0.350667,0.0,0.120103,0.0,...,0.029160,0.0,0.229049,0.0,0.048338,0.0,0.117346,0.0,0.148204,0
1,0.060056,0.0,0.300080,0.0,0.118237,0.0,0.352791,0.0,0.120099,0.0,...,0.031028,0.0,0.231224,0.0,0.048061,0.0,0.117367,0.0,0.148350,0
2,0.060178,0.0,0.301804,0.0,0.118667,0.0,0.354974,0.0,0.120154,0.0,...,0.032944,0.0,0.233464,0.0,0.047822,0.0,0.117458,0.0,0.148564,0
3,0.060323,0.0,0.303663,0.0,0.119179,0.0,0.357215,0.0,0.120268,0.0,...,0.034909,0.0,0.235768,0.0,0.047621,0.0,0.117620,0.0,0.148844,0
4,0.060494,0.0,0.305652,0.0,0.119772,0.0,0.359511,0.0,0.120438,0.0,...,0.036919,0.0,0.238133,0.0,0.047456,0.0,0.117850,0.0,0.149189,0


In [62]:
tonic_reshaped = reshape_function(tonic, name='Tonic')

In [63]:
tonic_reshaped.shape

(2106, 482)

In [64]:
tonic_reshaped.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_470,sample_471,sample_472,sample_473,sample_474,sample_475,sample_476,sample_477,sample_478,sample_479
0,S10,1.0,0.085659,0.085695,0.085731,0.085767,0.085804,0.085841,0.085878,0.085915,...,0.090835,0.090845,0.090855,0.090865,0.090874,0.090884,0.090893,0.090902,0.090911,0.090919
1,S10,1.0,0.088932,0.088947,0.088963,0.088978,0.088993,0.089008,0.089023,0.089039,...,0.089758,0.089745,0.089732,0.089720,0.089708,0.089697,0.089687,0.089677,0.089667,0.089659
2,S10,1.0,0.090427,0.090435,0.090443,0.090450,0.090458,0.090465,0.090473,0.090480,...,0.090864,0.090881,0.090899,0.090916,0.090933,0.090951,0.090968,0.090986,0.091004,0.091022
3,S10,1.0,0.090325,0.090322,0.090318,0.090315,0.090313,0.090310,0.090307,0.090305,...,0.093967,0.094001,0.094036,0.094070,0.094105,0.094140,0.094175,0.094209,0.094244,0.094279
4,S10,1.0,0.090928,0.090936,0.090944,0.090952,0.090959,0.090966,0.090973,0.090980,...,0.094783,0.094738,0.094693,0.094648,0.094603,0.094557,0.094511,0.094464,0.094418,0.094371


Tonic descriptives row-wise for max and mean

In [65]:
# intend to apply this function to phasic reshaped
def rise_time_std(row, sampling_rate = fs):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        # Calculate peak count
        peaks = info["SCR_Peaks"]
        peak_count = len(peaks)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        rise_time_std = np.nanstd(rise_times)
        return pd.Series(
                    [peak_count, rise_time_std],
                    index=["phasic_peak_count", "phasic_std_rise_time"],
                )

    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan

In [66]:
def sig2features(sig, sig_name):
    """
    sig is a signal, like tonic_reshaped
    sig_name is a string, like "tonic"
    #feature_set is a list, like [min, max]
    """
    # only want the signals, no "subject_id" and "label" columns
    sample_cols = [col for col in sig.columns if col.startswith('sample_')]
    sig_descriptives = pd.DataFrame()
    # Bring Labels into descriptives to the database
    sig_descriptives['label'] = sig['label']
    # Bring subject ID over to descriptives
    if 'subject_id' in sig.columns:
        sig_descriptives['subject_id'] = sig['subject_id']
    # Descriptives
    sig_descriptives[f"{sig_name}_mean"] = sig[sample_cols].mean(axis=1)
    sig_descriptives[f"{sig_name}_max"]  = sig[sample_cols].max(axis=1)
    sig_descriptives[f"{sig_name}_std"]  = sig[sample_cols].std(axis=1)
    sig_descriptives[f"{sig_name}_skew"]  = sig[sample_cols].skew(axis=1)
    sig_descriptives[f"{sig_name}_norm1"]  = sig[sample_cols].abs().sum(axis=1)
    if sig_name == 'phasic':
        sig_descriptives[["phasic_peak_count", "phasic_std_rise_time"]] = sig[sample_cols].apply(rise_time_std, axis=1)
    return sig_descriptives

In [67]:
tonic_descriptives = sig2features(tonic_reshaped, 'tonic')
tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max,tonic_std,tonic_skew,tonic_norm1
0,1.0,S10,0.089573,0.090919,0.001314,-1.351067,42.994987
1,1.0,S10,0.090317,0.091041,0.000479,-0.892952,43.352218
2,1.0,S10,0.090413,0.091041,0.000387,-0.491455,43.398353
3,1.0,S10,0.090903,0.094279,0.001094,1.341259,43.633596
4,1.0,S10,0.092115,0.095903,0.002147,0.564912,44.215323


Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [68]:
# Test example before I implemented on full dataframe the rise time standard deviation descriptive
phasic_reshape = reshape_function(phasic, name='Phasic')
row_example = phasic_reshape.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=fs)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / fs
np.nanstd(rise_times_sec)


np.float64(1.0380269746013349)

In [69]:

phasic_descriptives = sig2features(phasic_reshape, 'phasic')
phasic_descriptives.head()

/opt/anaconda3/envs/py314/lib/python3.14/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/opt/anaconda3/envs/py314/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/py314/lib/python3.14/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/opt/anaconda3/envs/py314/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/py314/lib/python3.14/site-packages

,label,subject_id,phasic_mean,phasic_max,phasic_std,phasic_skew,phasic_norm1,phasic_peak_count,phasic_std_rise_time
0,1.0,S10,0.002554,0.562885,0.028720,16.633820,1.226123,5.0,1.038027
1,1.0,S10,0.000303,0.061246,0.003327,14.854241,0.145350,4.0,3.506442
2,1.0,S10,0.000072,0.019111,0.001117,15.723381,0.034454,1.0,0.000000
3,1.0,S10,0.000088,0.019111,0.001171,14.195701,0.042252,2.0,0.000000
4,1.0,S10,0.000016,0.007792,0.000356,21.908896,0.007821,1.0,0.000000


Create and save file descriptive file

In [70]:
eda_reshaped = reshape_function(eda, name='EDA')
eda_descriptives = sig2features(eda_reshaped, 'eda')
eda_descriptives.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514


In [71]:
feature_dataframe = pd.concat(
    [eda_descriptives,
     phasic_descriptives.drop(columns=["subject_id", "label"]), 
     tonic_descriptives.drop(columns=["subject_id", "label"]),
     ],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1,phasic_mean,phasic_max,phasic_std,phasic_skew,phasic_norm1,phasic_peak_count,phasic_std_rise_time,tonic_mean,tonic_max,tonic_std,tonic_skew,tonic_norm1
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615,0.002554,0.562885,0.028720,16.633820,1.226123,5.0,1.038027,0.089573,0.090919,0.001314,-1.351067,42.994987
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551,0.000303,0.061246,0.003327,14.854241,0.145350,4.0,3.506442,0.090317,0.091041,0.000479,-0.892952,43.352218
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095,0.000072,0.019111,0.001117,15.723381,0.034454,1.0,0.000000,0.090413,0.091041,0.000387,-0.491455,43.398353
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344,0.000088,0.019111,0.001171,14.195701,0.042252,2.0,0.000000,0.090903,0.094279,0.001094,1.341259,43.633596
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514,0.000016,0.007792,0.000356,21.908896,0.007821,1.0,0.000000,0.092115,0.095903,0.002147,0.564912,44.215323


In [72]:
feature_dataframe.shape

(2106, 19)

In [73]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
